# Confronto europeo

Confronti completi con la sezione `confronto_europeo`: pressione, spesa pubblica,
spesa sociale e dettaglio COFOG.


## Scaricamento dati

Esegui questa cella prima degli import di `utils_bilancio`. La cella usa solo Python standard per trovare il repository e, se serve, lancia `scripts/run_sections.py` per creare o aggiornare il JSON della sezione.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

SECTION_ID = "confronto_europeo"
REFRESH = False
FORCE_DOWNLOAD = False
repo_root = None
for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (candidate / "scripts" / "utils_bilancio").is_dir():
        repo_root = candidate
        break

if repo_root is None:
    raise ModuleNotFoundError("Impossibile trovare il repository Bilancio_pubblico.")

section_file = repo_root / "data" / "export" / "bilancio-pubblico" / "sections" / f"{SECTION_ID}.json"
command = [sys.executable, str(repo_root / "scripts" / "run_sections.py"), "--sections", SECTION_ID]
if REFRESH or FORCE_DOWNLOAD:
    command.append("--refresh")

env = os.environ.copy()
if SECTION_ID == "regioni" and SIOPE_YEARS:
    env["BILANCIO_PUBBLICO_SIOPE_YEARS"] = SIOPE_YEARS

if FORCE_DOWNLOAD or REFRESH or not section_file.exists():
    print("Eseguo:", " ".join(command))
    subprocess.run(command, cwd=repo_root, check=True, env=env)
else:
    print(f"Uso cache: {section_file}")


## Import e caricamento

Dopo che i file sono presenti, questa cella importa gli helper del repo e carica `section`, `status`, `frame` e l'eventuale `source_payload`.


In [ ]:
from pathlib import Path
import sys

SECTION_ID = "confronto_europeo"

if "repo_root" not in globals() or repo_root is None:
    for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (candidate / "scripts" / "utils_bilancio").is_dir():
            repo_root = candidate
            break
    else:
        raise ModuleNotFoundError("Impossibile trovare il repository Bilancio_pubblico.")

scripts_dir = repo_root / "scripts"
if str(scripts_dir) not in sys.path:
    sys.path.insert(0, str(scripts_dir))

from utils_bilancio.notebook.utils import setup_notebook_section

notebook_state = setup_notebook_section(
    section_id=SECTION_ID,
    refresh=False,
    force_download=False,
    include_source_payload=SECTION_ID == "italia",
)

section = notebook_state["section"]
status = notebook_state["status"]
frame = notebook_state["frame"]
source_payload = notebook_state.get("source_payload")


## Parametri principali

Nella cella **Scaricamento dati** puoi modificare:
- `REFRESH = False/True`: aggiorna Eurostat e rigenera la sezione.
- `FORCE_DOWNLOAD = False/True`: forza una nuova materializzazione anche se il JSON esiste gia'.

Nelle celle di analisi puoi regolare:
- `METRIC`: usa uno dei valori stampati in `EU_METRICS`, ad esempio `"tax_pressure"`, `"public_spending"`, `"social_spending"`.
- `YEAR`: anno fra quelli stampati in `EU_YEAR_OPTIONS`; `None` usa l'ultimo disponibile.
- `TOP`: numero di paesi da mostrare, di solito tra `5` e `30`.
- `COUNTRIES`: lista di paesi esattamente come stampati in `EU_COUNTRY_OPTIONS`.
- `COFOG_CODE`: codice fra quelli stampati in `EU_COFOG_CODES`, ad esempio `"GF10"` o `"GF0101"`.

I nomi paese, i codici COFOG e le metriche sono case-sensitive: copia i valori dalla cella opzioni.


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 170)
plt.style.use("ggplot")



In [ ]:
def rows_to_df(rows):
    return frame(rows or [])


def latest_year(df, year_col="tax_year"):
    if df.empty:
        return None
    for key in ("tax_year", "spending_year", "social_year", "year"):
        if key in df.columns:
            values = pd.to_numeric(df[key], errors="coerce").dropna()
            if not values.empty:
                return int(values.max())
    return None


def plot_peer_metric(df, metric, top=20):
    if df.empty or metric not in df.columns:
        print("Nessun dato")
        return pd.DataFrame()

    year_key = "tax_year" if metric == "tax_pressure" else "spending_year" if metric == "public_spending" else "social_year"
    year = latest_year(df, year_key)
    filtered = df
    if year is not None and year_key in df.columns:
        filtered = df[df[year_key] == year]

    ordered = filtered.dropna(subset=[metric]).sort_values(metric, ascending=False)
    if ordered.empty:
        print("Nessun dato")
        return pd.DataFrame()

    ordered.head(top).plot(x="country", y=metric, kind="bar", figsize=(12, 5), legend=False)
    plt.title(f"{metric} - top {top}")
    plt.tight_layout()
    plt.show()
    return ordered


def plot_scatter_against_italy(df, metric, year):
    if df.empty or metric not in df.columns:
        print("Dati insufficienti")
        return
    year_col = "tax_year" if metric == "tax_pressure" else "spending_year" if metric == "public_spending" else "social_year"
    selected = df.copy()
    if year_col in df.columns:
        selected = selected[selected[year_col] == year]
    selected = selected.dropna(subset=[metric, "country"])
    selected = selected.sort_values(metric)
    selected[metric] = pd.to_numeric(selected[metric], errors="coerce")


def plot_cofog_code(function_rows, code, year=None, top=20):
    if function_rows.empty:
        print("Nessuna riga COFOG")
        return
    data = function_rows
    if year is not None and "year" in data.columns:
        data = data[data["year"] == year]
    data = data[data["cofog_code"] == code].dropna(subset=["value"])
    if data.empty:
        print("Nessun dato per il codice")
        return
    data = data.sort_values("value", ascending=False).head(top)
    data.plot(x="country", y="value", kind="bar", figsize=(12, 5), legend=False)
    plt.title(f"COFOG {code}")
    plt.tight_layout()
    plt.show()


## Elenco opzioni disponibili

Esegui questa cella prima delle celle dove imposti parametri:

- controlla i nomi ammessi
- usa esattamente i valori indicati
- verifica i codici/regioni/anni disponibili nella tua versione dati


In [ ]:
# Opzioni confronto europeo - parametri ammessi

def print_values(header, values):
    print(header)
    if not values:
        print("  (nessun valore)")
        return
    for item in values:
        print(f" - {item}")


def format_lines(values, per_line=10):
    values = [str(item) for item in values]
    if not values:
        return ["(vuoto)"]
    values = sorted(set(values))
    return [", ".join(values[i : i + per_line]) for i in range(0, len(values), per_line)]

peer_rows = rows_to_df(section.get("peer", []))
peer_series = rows_to_df(section.get("peer_series", []))
function_options = rows_to_df(section.get("peer_spending_function_options", []))
function_rows = rows_to_df(section.get("peer_spending_functions", []))
function_series = rows_to_df(section.get("peer_spending_functions_series", []))

PEER_COUNTRIES = []
for df in [peer_rows, peer_series, function_rows, function_series]:
    if not df.empty and "country" in df.columns:
        PEER_COUNTRIES.extend(df["country"].dropna().astype(str).tolist())
PEER_COUNTRIES = sorted(set(PEER_COUNTRIES))

metric_labels = rows_to_df(section.get("metrics", []))
if not metric_labels.empty and "id" in metric_labels.columns:
    PEER_METRICS = sorted(set(metric_labels["id"].dropna().astype(str).tolist()))
else:
    PEER_METRICS = ["tax_pressure", "public_spending", "social_spending"]

PEER_YEAR_OPTIONS = {}
if not peer_series.empty and {"metric_id", "year"}.issubset(peer_series.columns):
    for metric in sorted(set(peer_series["metric_id"].dropna().astype(str).tolist())):
        years = peer_series.loc[peer_series["metric_id"] == metric, "year"]
        years = pd.to_numeric(years, errors="coerce").dropna().astype(int).sort_values().unique().tolist()
        PEER_YEAR_OPTIONS[metric] = years
else:
    year_lookup = {
        "tax_pressure": "tax_year",
        "public_spending": "spending_year",
        "social_spending": "social_year",
    }
    for metric in PEER_METRICS:
        year_col = year_lookup.get(metric)
        if not peer_rows.empty and year_col in peer_rows.columns:
            years = pd.to_numeric(peer_rows[year_col], errors="coerce").dropna().astype(int).sort_values().unique().tolist()
            PEER_YEAR_OPTIONS[metric] = years

if not function_options.empty and "id" in function_options.columns:
    PEER_COF = sorted(set(function_options["id"].dropna().astype(str).tolist()))
else:
    PEER_COF = []

function_labels = []
if not function_options.empty and "label" in function_options.columns:
    function_labels = sorted(set(function_options["label"].dropna().astype(str).tolist()))

cofog_years = []
if not function_series.empty and "year" in function_series.columns:
    cofog_years = pd.to_numeric(function_series["year"], errors="coerce").dropna().astype(int).sort_values().unique().tolist()

print("=== Parametri disponibili (Confronto europeo) ===")
print_values("Paesi disponibili:", PEER_COUNTRIES)
print(f"Metriche disponibili per METRIC: {PEER_METRICS}")
print(f"Anni disponibili per metrica: {PEER_YEAR_OPTIONS}")
print(f"Anni disponibili per COFOG storico: {cofog_years}")
print("Top consigliato: da 5 a 30")

print()
print("Codici COFOG disponibili:")
print_values("Totale codici: %s" % len(PEER_COF), PEER_COF)

print()
print("Etichette COFOG (prime 120):")
if function_labels:
    for line in format_lines(function_labels[:120], per_line=8):
        print(f" - {line}")
else:
    print("  (vuoto)")

print()
print("Esempi parametri:")
print("METRIC = 'public_spending'")
print("YEAR = None")
print("COUNTRIES = ['Italia', 'Francia', 'Germania', 'Spagna']")
print("COFOG_CODE = 'GF10'")
print("Nota: i parametri sono case-sensitive: usa esattamente questi valori nella cella di confronto.")

EU_COUNTRY_OPTIONS = PEER_COUNTRIES
EU_METRICS = PEER_METRICS
EU_YEAR_OPTIONS = PEER_YEAR_OPTIONS
EU_COFOG_CODES = PEER_COF
EU_COFOG_LABELS = function_labels
EU_COFOG_YEAR_OPTIONS = cofog_years


## Stato sezione

Conta i blocchi disponibili e verifica i range temporali.

In [ ]:
peer_rows = rows_to_df(section.get("peer", []))
function_options = rows_to_df(section.get("peer_spending_function_options", []))
function_rows = rows_to_df(section.get("peer_spending_functions", []))

print("Peer presenti:", len(peer_rows))
print("COFOG option:", len(function_options), "righe COFOG:", len(function_rows))
print("Anni disponibili:", section.get("latest_years", {}))

if not peer_rows.empty:
    display(peer_rows.sort_values("tax_pressure", ascending=False).head(10))
if not function_options.empty:
    display(function_options.head(8))


## Confronto principale

Scegli METRIC, YEAR e TOP.

In [ ]:
METRIC = "tax_pressure"
YEAR = None
TOP = 18

selected = plot_peer_metric(peer_rows, METRIC, top=TOP)

for metric in ["public_spending", "social_spending", "tax_pressure"]:
    plot_peer_metric(peer_rows, metric, top=12)

if not peer_rows.empty and YEAR is None:
    YEAR = latest_year(peer_rows, "tax_year")

if YEAR is not None and not peer_rows.empty:
    peer_year = peer_rows.copy()
    peer_year = peer_year[peer_year[["tax_year", "spending_year", "social_year"]].fillna(-1).max(axis=1) == peer_year[["tax_year", "spending_year", "social_year"]].fillna(-1).max(axis=1)]


## COFOG confronti

Confronti per funzione COFOG e livello padre/codice.

In [ ]:
if function_options.empty:
    print("Opzioni COFOG non presenti")
else:
    print("Campione codici livello 1:")
    print(function_options[function_options["level"] == 1][["id", "label"]].head(15).to_string(index=False))

COFOG_CODE = "GF01"
current_year = YEAR if YEAR is not None else latest_year(function_rows, "year")
if current_year is None:
    current_year = latest_year(function_rows, "year")
plot_cofog_code(function_rows, COFOG_CODE, year=current_year, top=12)

# media per codice COFOG nell'anno corrente
if not function_rows.empty and "cofog_label" in function_rows.columns:
    target_year = current_year or int(function_rows["year"].max())
    pivot = function_rows[function_rows["year"] == target_year]
    ranked = pivot.groupby("cofog_label")["value"].mean().sort_values(ascending=False).head(15)
    if not ranked.empty:
        ranked.sort_values().plot(kind="barh", figsize=(12, 6))
        plt.title(f"Spesa COFOG media {target_year}")
        plt.tight_layout()
        plt.show()


## Esplorazione multi-metrica

Confronti per top e bottom rapidamente per più metriche.

In [ ]:
if not peer_rows.empty:
    for metric in ["tax_pressure", "public_spending", "social_spending"]:
        year_key = "tax_year" if metric == "tax_pressure" else "spending_year" if metric == "public_spending" else "social_year"
        if year_key in peer_rows.columns:
            year = int(pd.to_numeric(peer_rows[year_key], errors="coerce").dropna().max())
        else:
            year = None
        data_year = peer_rows.copy()
        if year is not None and year_key in data_year.columns:
            data_year = data_year[data_year[year_key] == year]
        top = data_year.sort_values(metric, ascending=False).head(12)
        bottom = data_year.sort_values(metric, ascending=True).head(12)
        print(f"\n{metric} ({year}) - top")
        if not top.empty:
            print(top[["country", metric]].to_string(index=False))
        print(f"{metric} ({year}) - bottom")
        if not bottom.empty:
            print(bottom[["country", metric]].to_string(index=False))


## Serie storiche europee

Queste celle usano le nuove serie `peer_series` e `peer_spending_functions_series`: puoi confrontare paesi nel tempo per pressione fiscale, spesa pubblica, protezione sociale e singole funzioni COFOG.


In [ ]:
METRIC = "public_spending"
COUNTRIES = ["Italia", "Francia", "Germania", "Spagna"]

peer_series = rows_to_df(section.get("peer_series", []))
if peer_series.empty:
    print("peer_series assente: rigenera la sezione confronto_europeo.")
else:
    selected = peer_series[peer_series["metric_id"] == METRIC].copy()
    selected = selected[selected["country"].isin(COUNTRIES)]
    if selected.empty:
        print("Nessuna serie per i parametri scelti.")
    else:
        pivot = selected.pivot_table(index="year", columns="country", values="value", aggfunc="mean").sort_index()
        display(pivot.tail(10))
        pivot.plot(figsize=(12, 6), marker="o")
        plt.title(f"{METRIC} - serie storica europea")
        plt.ylabel("% PIL")
        plt.tight_layout()
        plt.show()


In [ ]:
COFOG_CODE = "GF10"
COUNTRIES = ["Italia", "Francia", "Germania", "Spagna"]

function_series = rows_to_df(section.get("peer_spending_functions_series", []))
if function_series.empty:
    print("Serie COFOG assenti: rigenera la sezione confronto_europeo.")
else:
    selected = function_series[function_series["cofog_code"] == COFOG_CODE].copy()
    selected = selected[selected["country"].isin(COUNTRIES)]
    if selected.empty:
        print("Nessuna serie COFOG per i parametri scelti.")
    else:
        pivot = selected.pivot_table(index="year", columns="country", values="value", aggfunc="mean").sort_index()
        display(pivot.tail(10))
        pivot.plot(figsize=(12, 6), marker="o")
        plt.title(f"COFOG {COFOG_CODE} - % PIL")
        plt.ylabel("% PIL")
        plt.tight_layout()
        plt.show()
